In [2]:
# 1. Instalação das dependências necessárias
!pip install --upgrade "click>=8.1.3,<8.2.0" -q
!pip install git+https://github.com/openai/whisper.git -q
!pip install -U google-genai -q
!pip install gTTS -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
import os
import whisper
import google.generativeai as genai
from gtts import gTTS
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [4]:
# Configurações iniciais
language = 'pt'

# 1. Gravação de Áudio Com Python e JavaScript 🎤

In [7]:
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  # Executa o código JavaScript para gravar o áudio
  display(Javascript(RECORD))
  # Recebe o áudio gravado como resultado do JavaScript
  js_result = output.eval_js('record(%s)' % (sec * 1000))
   # Decodifica o áudio em base64
  audio = b64decode(js_result.split(',')[1])
  # Salva o áudio em um arquivo
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  # Retorna o caminho do arquivo de áudio (pasta padrão do Google Colab)
  return f'/content/{file_name}'

# Grava o áudio do usuário por um tempo determinado (padrão 5 segundos)
print('Ouvindo...\n')
record_file = record()

# Exibe o áudio gravado
display(Audio(record_file, autoplay=False))

Ouvindo...



<IPython.core.display.Javascript object>

# 2. Reconhecimento de Fala com Whisper 🧠

In [8]:
print('Transcrevendo com Whisper...\n')
# Selecione o modelo do Whisper que melhor atenda às suas necessidades:
# https://github.com/openai/whisper#available-models-and-languages
model = whisper.load_model("base")

# Transcreve o audio gravado anteriormente.
result = model.transcribe(record_file, fp16=False, language=language)
transcription = result["text"]
print(transcription)

Transcrevendo com Whisper...



100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 168MiB/s]


 O que é a inteligência artificial?


# 3. Integração com a API do Google Gemini 💬

In [9]:
# Obtenha sua chave gratuita em: https://aistudio.google.com/
from google.colab import userdata
import google.generativeai as genai # Adicionada esta linha para garantir o import correto

# Busca a chave de forma segura dos 'Secrets' do Colab
try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY) # Use genai.configure para definir a chave globalmente
    print("✅ Chave de API carregada com segurança!")
except Exception as e:
    print("❌ Erro: Certifique-se de que a chave 'GEMINI_API_KEY' está configurada nos Secrets do Colab.")

print('\nConsultando Gemini...\n')

try:
    # Criando o modelo. Usaremos 'gemini-2.5-flash' conforme listado nos modelos disponíveis.
    model_instance = genai.GenerativeModel('gemini-2.5-flash', system_instruction="Não use listas longas, tabelas ou caracteres especiais (como asteriscos para negrito). Use pontuação clara para criar pausas naturais na fala.")

    # Geramos a resposta com base na transcrição do Whisper
    response = model_instance.generate_content(transcription)

    gemini_response = response.text  # Mantendo o nome da variável original
    print(f"🤖 Gemini respondeu: {gemini_response}")

except Exception as e:
    print(f"❌ Erro ao gerar resposta: {e}")
    gemini_response = "Desculpe, tive um problema ao processar sua pergunta."

# # Listar os modelos disponíveis (mantido para referência, pode ser removido depois)
# print("Modelos Gemini disponíveis:")
# for m in genai.list_models():
#   if "generateContent" in m.supported_generation_methods:
#     print(m.name)

✅ Chave de API carregada com segurança!

Consultando Gemini...

🤖 Gemini respondeu: A inteligência artificial, ou IA, é a capacidade de máquinas e sistemas de computador de simular a inteligência humana. Isso envolve a habilidade de aprender com dados, raciocinar, resolver problemas, reconhecer padrões, compreender a linguagem natural e até mesmo tomar decisões, tudo isso com base em algoritmos complexos.

Você a encontra em assistentes de voz como a Siri, recomendações de filmes na Netflix, carros autônomos, sistemas de reconhecimento facial e até nos filtros de spam do seu e-mail. O objetivo principal da IA é automatizar tarefas, otimizar processos e aprimorar as capacidades humanas, tornando os sistemas mais eficientes e inteligentes. É um campo em constante evolução, que busca replicar e até superar certas habilidades cognitivas que antes eram exclusivas dos seres humanos.


# 4. Sintetizando a Resposta do Gemini Como Voz (gTTS) 🔊

In [10]:
# Cria um objeto gTTS com a resposta gerada pelo Gemini e a língua que será sintetizada em voz (variável "language").
gtts_object = gTTS(text=gemini_response, lang=language, slow=False)

# Salva o áudio da resposta no arquivo especificado (pasta padrão do Google Colab)
response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)

# Reproduz o áudio da resposta salvo no arquivo
display(Audio(response_audio, autoplay=True))